# 02 · Replication of Tsantilis et al. (2025)

Replicates *Evaluating cross-series pooling protocols in intermittent demand forecasting: a factorial study*
(Tsantilis, Giannopoulos, Dasaklis & Patsakis, IEEE 2025; `2026248926.pdf`). The paper crosses three
architectures with two training regimes on the same 100 intermittent M5 series:

| | per-series regime (PS) | global regime (GL) |
|---|---|---|
| **LightGBM** (squared error) | LGBM-MSE-PS | LGBM-MSE-GL |
| **LSTM** | LSTM-PS | LSTM-GL |
| **Reservoir computing** (echo state network) | RES-PS | RES-GL |

Everything else is held fixed: the input is raw lagged demand only (no calendar, price or series features),
every model forecasts all 28 days in one direct step, each cell is tuned with Hyperopt TPE (100 evaluations),
the top-3 configurations are refit and averaged, and every cell is scored with RMSSE@28 on the blind window
`d_1914`–`d_1941`.

**Note on the split.** The paper uses training `d_1`–`d_1885`, validation `d_1886`–`d_1913` and blind test
`d_1914`–`d_1941`. This differs from the thesis split used elsewhere (test `d_1942`–`d_1969`) and is kept here so
that the numbers are comparable with the paper.

**Outputs**
- sample of 100 series: `results/replication/sample.csv`
- per-run forecasts and Hyperopt trials: `results/replication/runs/<run>/`
- tables: `results/replication/*.csv`; figures: `figures/replication/*.png` and `*.pdf`

### What the paper does not specify

The paper leaves several implementation details open. The choices made here are listed so that they can be
reported in the thesis and revisited in the extension notebook.

| Topic | Paper | This replication |
|---|---|---|
| Selection window | not stated | `d_1`–`d_1913`, statistics over each series' active period (first sale onwards); the blind window is never used |
| ADF screen | stationarity test | constant term, AIC lag selection, keep series with p < 0.05 |
| Ljung–Box screen | white-noise screen at lags 13, 26, 52 | drop a series if no lag is significant at 0.05 |
| Regime cells | eight populated cells of zero proportion, ADI and SBC class | zero proportion over `d_1`–`d_1913` (< 60%, 60–80%, ≥ 80%) × ADI (≤ 2, 2–4, > 4); the SBC class is constant after stage 4. 8 of the 9 cells are populated |
| Sampling seed | seeded, value not stated | 42 |
| Input window | not stated (≈ 1.8K windows per series) | 56 days, giving 1,802 training windows per series from `d_1`–`d_1885` |
| Leading zeros | only skipped in the RMSSE scale | windows start at `d_1`, so pre-introduction zeros are part of the training data |
| Search spaces | architecture-specific, not listed | Section 2 |
| LSTM training | Linear(h, 28) head | Adam, MSE loss, gradient clipping at 1, dropout on the last hidden state |
| ESN | Ridge readout on the reservoir state at the end of the window | leaky ESN, zero initial state for every window, Ridge with intercept |
| Model seeds | seeded | 42 for Hyperopt, LightGBM, PyTorch and the reservoir |

In [ ]:
import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import experiment as ex
import factorial_stats as fstats
import m5_data as m5
from intermittency import ADI_CUTOFF, CV2_CUTOFF, demand_stats
from plot_style import AXIS, CONTEXT, INK, INK_SECONDARY, MUTED, SERIES, apply_style, savefig
from series_sampling import LB_LAGS, hamilton_allocation, screen_series, stratified_sample

apply_style()
pd.set_option("display.float_format", "{:,.4f}".format)

RESULTS = ex.RESULTS_DIR
RESULTS.mkdir(parents=True, exist_ok=True)
FIG = "replication"

# Hue by architecture (validated palette); line style and marker by regime, so identity never relies on colour alone.
ARCH_COLORS = {"lgbm": SERIES[0], "lstm": SERIES[1], "esn": SERIES[2]}
CELL_COLORS = {cell: ARCH_COLORS[arch] for cell, (arch, _) in ex.CELLS.items()}
CELL_LINE = {cell: "-" if regime == "ps" else "--" for cell, (_, regime) in ex.CELLS.items()}
CELL_MARKER = {cell: "o" if regime == "ps" else "s" for cell, (_, regime) in ex.CELLS.items()}

## 1. Sample selection

Four-stage filter over all 30,490 item-store series, then a stratified sample of exactly 100 series. The
ADF and Ljung–Box screens take a few minutes and are cached in `data/processed/series_screen_train.parquet`.

In [ ]:
SELECT_END = ex.TRAIN_END  # d_1 - d_1913: the blind window is not used for selection
ALPHA = 0.05
SAMPLE_SIZE = 100
SAMPLE_SEED = 42
ZERO_EDGES, ZERO_LABELS = [0, 0.6, 0.8, 1.0], ["<60%", "60-80%", ">=80%"]
ADI_EDGES, ADI_LABELS = [ADI_CUTOFF, 2, 4, np.inf], ["<=2", "2-4", ">4"]

sales = m5.load_sales(last_day=SELECT_END)
Y_sel = sales[m5.day_cols(1, SELECT_END)].to_numpy()
stats = demand_stats(Y_sel)  # over the active period (first sale to d_1913)
stats.insert(0, "id", sales["id"].astype(str).to_numpy())
stats["zero_prop"] = (Y_sel == 0).mean(axis=1)  # whole window, leading zeros included

screen_file = m5.PROCESSED_DIR / "series_screen_train.parquet"
if screen_file.exists():
    screen = pd.read_parquet(screen_file)
else:
    screen = screen_series(Y_sel, stats.first_sale.to_numpy())
    screen.insert(0, "id", stats["id"])
    screen.to_parquet(screen_file, index=False)
assert (screen["id"].astype(str).to_numpy() == stats["id"].to_numpy()).all()
stats = pd.concat([stats, screen.drop(columns="id")], axis=1)

lb_cols = [f"lb_p{lag}" for lag in LB_LAGS]
stages = {
    "All item-store series": np.ones(len(stats), bool),
    "1. ADF: unit root rejected (p < 0.05)": stats.adf_p < ALPHA,
    "2. Ljung-Box: autocorrelation at lag 13, 26 or 52": (stats[lb_cols] < ALPHA).any(axis=1),
    "3. SBC class defined": stats.adi.notna() & stats.cv2.notna(),
    "4. Intermittent quadrant (ADI > 1.32, CV² <= 0.49)": (stats.adi > ADI_CUTOFF) & (stats.cv2 <= CV2_CUTOFF),
}
keep = np.ones(len(stats), bool)
funnel = {}
for name, mask in stages.items():
    keep &= np.asarray(mask)
    funnel[name] = keep.sum()
funnel = pd.Series(funnel, name="series").to_frame()
funnel["share_of_all"] = funnel.series / len(stats)
funnel.to_csv(RESULTS / "selection_funnel.csv")
population = stats[keep].copy()
funnel.style.format({"series": "{:,}", "share_of_all": "{:.1%}"})

In [ ]:
zero_band = pd.cut(population.zero_prop, ZERO_EDGES, right=False, labels=ZERO_LABELS)
adi_band = pd.cut(population.adi, ADI_EDGES, labels=ADI_LABELS)
assert zero_band.notna().all() and adi_band.notna().all()
cell_labels = [f"zero {z} | ADI {a}" for z in ZERO_LABELS for a in ADI_LABELS]
population["cell"] = pd.Categorical(
    [f"zero {z} | ADI {a}" for z, a in zip(zero_band, adi_band)], categories=cell_labels
).remove_unused_categories()

counts = population.cell.value_counts().sort_index()
allocation = pd.DataFrame(
    {"population": counts, "quota": counts / counts.sum() * SAMPLE_SIZE, "sample": hamilton_allocation(counts, SAMPLE_SIZE)}
)
sample = population.loc[stratified_sample(population.cell, SAMPLE_SIZE, SAMPLE_SEED)]
assert len(sample) == SAMPLE_SIZE and (sample.cell.value_counts().sort_index() == allocation["sample"]).all()
allocation.to_csv(RESULTS / "sample_allocation.csv")

print(f"Populated regime cells: {len(counts)} of {len(cell_labels)}")
print(pd.crosstab(zero_band, adi_band).rename_axis(index="zero proportion", columns="ADI").to_string(), "\n")
allocation.style.format({"population": "{:,}", "quota": "{:.2f}"})

In [ ]:
sample_cols = ["id", "cell", "first_sale", "n_active", "zero_prop", "zero_share", "adi", "cv2", "adf_p", *lb_cols]
sample[sample_cols].to_csv(ex.SAMPLE_FILE, index=False)

describe = pd.concat(
    {
        "Population": population[["first_sale", "zero_prop", "zero_share", "adi", "cv2"]].median(),
        "Sample": sample[["first_sale", "zero_prop", "zero_share", "adi", "cv2"]].median(),
    },
    axis=1,
).rename(index={"first_sale": "first sale (day)", "zero_prop": "zero proportion, d_1-d_1913",
                "zero_share": "zero share, active period", "adi": "ADI", "cv2": "CV²"})
print(f"Saved {len(sample)} series to {ex.SAMPLE_FILE.relative_to(ROOT)}; "
      f"categories: {sample.id.str.split('_').str[0].value_counts().to_dict()}")
describe.rename_axis("median").style.format("{:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.scatter(population.adi, population.zero_prop, s=3, color=CONTEXT, alpha=0.35, lw=0, rasterized=True,
           label=f"Filtered population ({len(population):,})")
ax.scatter(sample.adi, sample.zero_prop, s=26, color=SERIES[0], edgecolor="white", lw=0.8,
           label=f"Stratified sample ({len(sample)})")
for z in ZERO_EDGES[1:-1]:
    ax.axhline(z, color=INK_SECONDARY, lw=0.8, ls=":")
for a in ADI_EDGES[1:-1]:
    ax.axvline(a, color=INK_SECONDARY, lw=0.8, ls=":")
ax.set_xscale("log")
ax.set_xticks([1.32, 2, 4, 10, 30, 100], ["1.32", "2", "4", "10", "30", "100"])
ax.minorticks_off()
ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.grid(axis="y", visible=False)
ax.set_title("Regime cells of the intermittent population and the sample")
ax.set_xlabel("ADI over the active period (log scale)")
ax.set_ylabel("Zero days, d_1-d_1913")
ax.legend(loc="lower right", markerscale=1.5)
savefig(fig, "sample_regime_cells", subdir=FIG)
plt.show()

## 2. Experimental design

Each series is min-max scaled to [0, 1] with a scaler fit on its own training data. A training window is
56 consecutive days of input and the following 28 days of target. The per-series regime fits one model per
series on its 1,802 windows. The global regime stacks the windows of all 100 series into one training set,
with no series identifier.

| Architecture | Hyperopt search space (identical in both regimes) |
|---|---|
| LightGBM | `num_leaves` 4–64 (step 4), `learning_rate` 0.01–0.3 (log), `n_estimators` 50–500 (step 25), `min_data_in_leaf` 5–100 (step 5), `feature_fraction` 0.5–1, `bagging_fraction` 0.5–1, `lambda_l2` 0.001–10 (log) |
| LSTM | `hidden_size` {16, 32, 64, 128}, `num_layers` {1, 2}, `dropout` 0–0.4, `learning_rate` 1e-4–1e-2 (log), `batch_size` {64, 128, 256, 512}, `epochs` 5–40 (step 5) |
| ESN | `n_reservoir` 50–500 (step 25), `spectral_radius` 0.1–1.5, `input_scaling` 0.01–5 (log), `leak_rate` 0.05–1, `density` 0.01–0.5, `bias_scaling` 0–1, `ridge_alpha` 1e-6–100 (log) |

The tuning objective is the validation RMSSE on `d_1886`–`d_1913`: per series in the PS regime, and the mean
over the 100 series in the GL regime. The three best configurations are refit on `d_1`–`d_1913`. Their blind
forecasts are clipped at zero and then averaged.

In [ ]:
ids, panel = ex.load_panel()
N_SERIES = int(os.environ.get("REPLICATION_N_SERIES", len(ids)))  # smaller values only for quick checks
ids, panel = ids[:N_SERIES], panel[:N_SERIES]

tune0, final0 = ex.prepare(panel[0], ex.TUNE_END), ex.prepare(panel[0], ex.TRAIN_END)
print(f"Series: {len(ids)}")
print(f"Training windows per series: {len(tune0.X):,} (tuning), {len(final0.X):,} (final refit)")
print(f"Pooled training windows (global regime): {len(tune0.X) * len(ids):,} (tuning), {len(final0.X) * len(ids):,} (final)")

## 3. Running the six cells

The full protocol is expensive on a CPU: 100 Hyperopt evaluations × 100 series for each per-series cell,
and 100 evaluations on the pooled data for each global cell. Results are cached per series and per cell, so
an interrupted run resumes where it stopped and a finished run is only loaded. The long run is best started
outside Jupyter:

```bash
python src/experiment.py --run full --cells all --max-evals 100 --n-jobs 6
```

`REPLICATION_RUN`, `REPLICATION_MAX_EVALS` and `REPLICATION_N_SERIES` (environment variables) select a smaller
run for a quick end-to-end check of this notebook.

In [ ]:
RUN = os.environ.get("REPLICATION_RUN", "full")
MAX_EVALS = int(os.environ.get("REPLICATION_MAX_EVALS", 100))  # paper: 100
N_JOBS = os.cpu_count()

for cell in ex.CELLS:
    if ex.load_cell(RUN, cell, ids) is None:
        t0 = time.perf_counter()
        ex.run_cell(cell, ids, panel, RUN, MAX_EVALS, N_JOBS)
        print(f"{cell}: {(time.perf_counter() - t0) / 60:.1f} min")

runs = {cell: ex.load_cell(RUN, cell, ids) for cell in ex.CELLS}
assert all(r["max_evals"] == MAX_EVALS for r in runs.values()), "cached results were made with another budget"
runtime = pd.DataFrame(
    {"regime": [ex.CELLS[c][1].upper() for c in runs], "fit_hours": [r["seconds"] / 3600 for r in runs.values()]},
    index=pd.Index(list(runs), name="cell"),
)
print(f"Run '{RUN}': {MAX_EVALS} Hyperopt evaluations per tuning run, {len(ids)} series")
print("PS: summed single-thread time over all series; GL: wall time with all cores")
runtime.style.format({"fit_hours": "{:.2f}"})

## 4. Per-cell performance

RMSSE uses the M5 convention: the scale is the mean squared one-step naive error over the training history
from the first non-zero day. RMSSE@h is computed on the first h days of the blind window.

In [ ]:
final = [ex.prepare(y, ex.TRAIN_END) for y in panel]
curves = {
    cell: np.stack([ex.rmsse_by_horizon(d.actual, f, d.scale) for d, f in zip(final, r["forecast"])])
    for cell, r in runs.items()
}
scores = pd.DataFrame({cell: c[:, -1] for cell, c in curves.items()}, index=pd.Index(ids, name="id"))
scores.to_csv(RESULTS / f"rmsse28_per_series_{RUN}.csv")

table1 = fstats.summary_table(scores)
table1.to_csv(RESULTS / f"table1_{RUN}.csv")
pd.concat({"Replication": table1, "Paper (Table I)": fstats.PAPER_TABLE1}, axis=1)

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3))
ypos = np.arange(len(table1))[::-1]
for y, cell in zip(ypos, table1.index):
    ours, paper = table1.loc[cell, "Mean"], fstats.PAPER_TABLE1.loc[cell, "Mean"]
    ax.plot([ours, paper], [y, y], color=AXIS, lw=1.2, zorder=1)
    ax.plot(paper, y, marker=CELL_MARKER[cell], ms=8, mfc="white", mec=CELL_COLORS[cell], mew=1.6, ls="none", zorder=2)
    ax.plot(ours, y, marker=CELL_MARKER[cell], ms=8, color=CELL_COLORS[cell], mec="white", mew=1.2, ls="none", zorder=3)
ax.set_yticks(ypos, table1.index)
ax.grid(axis="y", visible=False)
ax.grid(axis="x", visible=True)
ax.set_title("Mean RMSSE@28 per cell: replication versus paper")
ax.set_xlabel("Mean RMSSE@28 over the blind series")
ax.legend(handles=[Line2D([], [], marker="o", ls="none", color=INK_SECONDARY, mec="white", ms=8, label="Replication"),
                   Line2D([], [], marker="o", ls="none", mfc="white", mec=INK_SECONDARY, mew=1.6, ms=8, label="Paper (Table I)")],
          loc="upper center", bbox_to_anchor=(0.5, -0.2), ncols=2)
savefig(fig, f"mean_rmsse_vs_paper_{RUN}", subdir=FIG)
plt.show()

In [ ]:
h = np.arange(1, ex.HORIZON + 1)
fig, ax = plt.subplots(figsize=(7, 3.4))
for cell, c in curves.items():
    ax.plot(h, c.mean(axis=0), color=CELL_COLORS[cell], ls=CELL_LINE[cell], marker=CELL_MARKER[cell],
            ms=3.5, markevery=[0, 6, 13, 20, 27], label=cell)
ax.set_xticks([1, 7, 14, 21, 28])
ax.set_xlim(0.5, 28.5)
ax.set_title("Mean RMSSE by forecast horizon")
ax.set_xlabel("Forecast horizon h (days)")
ax.set_ylabel("Mean RMSSE@h")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncols=3)
savefig(fig, f"rmsse_by_horizon_{RUN}", subdir=FIG)
plt.show()
pd.DataFrame({cell: c.mean(axis=0) for cell, c in curves.items()}, index=pd.Index(h, name="h")).to_csv(
    RESULTS / f"rmsse_by_horizon_{RUN}.csv")

## 5. Statistical analysis

**Planned contrasts** (paper Table II): the pooling effect within each architecture and the architecture
effect within the global regime, tested with two-sided Wilcoxon signed-rank tests and Holm-corrected within
the family of six. A positive mean(A − B) means that A has the higher (worse) RMSSE@28.

In [ ]:
planned = fstats.wilcoxon_holm(scores, fstats.PLANNED_CONTRASTS)
table2 = planned.join(fstats.PAPER_TABLE2.add_prefix("paper_"))
table2.to_csv(RESULTS / f"table2_planned_contrasts_{RUN}.csv")
table2.drop(columns=["A", "B"])

**All 15 pairs** (paper Table III), Holm-corrected over the family of 15.

In [ ]:
pairwise = fstats.wilcoxon_holm(scores)
table3 = pairwise.reset_index(drop=True).merge(fstats.PAPER_TABLE3, on=["A", "B"], suffixes=("", "_paper"))
table3.to_csv(RESULTS / f"table3_pairwise_{RUN}.csv", index=False)
print(f"Significant pairs (Holm p < 0.05): replication {(table3.p_holm < 0.05).sum()}, "
      f"paper {(table3.p_holm_paper < 0.05).sum()}")
table3

**Friedman and Nemenyi.** The Friedman test checks whether the six cells perform equally. The Nemenyi critical
difference (CD) gives the smallest mean-rank gap that counts as significant at α = 0.05. Rank 1 means the
lowest RMSSE@28 on a series. In the diagram, cells joined by a bar are not significantly different.

In [ ]:
fr = fstats.friedman_nemenyi(scores)
print(f"Friedman: chi2 = {fr['chi2']:.2f}, p = {fr['p']:.6f}   (paper: chi2 = {fstats.PAPER_FRIEDMAN['chi2']}, "
      f"p = {fstats.PAPER_FRIEDMAN['p']})")
print(f"Nemenyi CD (alpha = 0.05, k = {scores.shape[1]}, N = {len(scores)}): {fr['cd']:.3f}   (paper: {fstats.PAPER_FRIEDMAN['cd']})")
ranks = pd.concat({"replication": fr["mean_ranks"], "paper": fstats.PAPER_MEAN_RANKS}, axis=1).rename_axis("mean rank")
ranks.to_csv(RESULTS / f"mean_ranks_{RUN}.csv")

fig, axes = plt.subplots(2, 1, figsize=(7, 4.6))
fstats.plot_cd_diagram(axes[0], fr["mean_ranks"], fr["cd"], CELL_COLORS, ink=INK, muted=MUTED)
fstats.plot_cd_diagram(axes[1], fstats.PAPER_MEAN_RANKS, fstats.PAPER_FRIEDMAN["cd"], CELL_COLORS, ink=INK, muted=MUTED)
axes[0].set_title(f"Replication (N = {len(scores)})", loc="left")
axes[1].set_title("Paper, Fig. 4 (N = 100)", loc="left")
fig.suptitle("Critical difference diagrams (Nemenyi, α = 0.05)", x=0.02, ha="left", fontweight="semibold")
fig.tight_layout()
savefig(fig, f"cd_diagram_{RUN}", subdir=FIG)
plt.show()
ranks

## 6. Does the replication confirm the paper?

Each row is one of the paper's main claims, evaluated on the replication results with the same test and
threshold.

In [ ]:
def contrast(name):
    return planned.loc[name, "mean_diff"], planned.loc[name, "p_holm"]


def pooling_helps(name):
    diff, p = contrast(name)
    return diff > 0 and p < ALPHA


claims = {
    "Pooling significantly improves the reservoir (RES)": pooling_helps("Pool. Reservoir (PS vs GL)"),
    "Pooling significantly improves LightGBM": pooling_helps("Pool. LightGBM (PS vs GL)"),
    "Pooling has no significant effect on the LSTM": contrast("Pool. LSTM (PS vs GL)")[1] >= ALPHA,
    "In GL, LSTM and RES differ significantly": contrast("Arch. in GL (LSTM vs RES)")[1] < ALPHA,
    "In GL, LightGBM and RES do not differ significantly": contrast("Arch. in GL (LGBM vs RES)")[1] >= ALPHA,
    "In GL, LightGBM and LSTM do not differ significantly": contrast("Arch. in GL (LGBM vs LSTM)")[1] >= ALPHA,
    "RES-GL has the lowest mean RMSSE@28": table1["Mean"].idxmin() == "RES-GL",
    "RES-GL has the best mean rank": fr["mean_ranks"].idxmin() == "RES-GL",
    "Friedman test rejects equal performance": fr["p"] < ALPHA,
}
verdict = pd.Series(claims, name="holds in the replication").rename_axis("claim of the paper").to_frame()
verdict.to_csv(RESULTS / f"verdict_{RUN}.csv")
print(f"{verdict.iloc[:, 0].sum()} of {len(verdict)} claims hold")
verdict

## Key findings

To be written once the full run (`RUN = "full"`, 100 evaluations per tuning run) has finished. The
extension notebook `03` builds on these results.